In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("Diabetes_prediction.csv")

In [ ]:
df.head()

In [ ]:
print("Shape:", df.shape)
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
feature_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
                "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
X = df[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Feature scaling applied using StandardScaler")
print("Scaled data shape:", X_scaled.shape)

In [ ]:
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(K_range), inertia, marker="o", color="blue", linewidth=2)
plt.title("Elbow Method - Finding Optimal K")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia (Within-cluster Sum of Squares)")
plt.xticks(list(K_range))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
silhouette_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}  Silhouette Score: {score:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker="s", color="green", linewidth=2)
plt.title("Silhouette Score vs Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.xticks(range(2, 11))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X_scaled)
df["Cluster"] = kmeans.labels_

print("Clustering complete (K=2)")
print("\nCluster counts:")
print(df["Cluster"].value_counts())

In [ ]:
print("Cluster vs Actual Diagnosis:")
print(df.groupby(["Cluster", "Diagnosis"]).size().unstack(fill_value=0))

In [ ]:
colors = ["blue" if c == 0 else "red" for c in df["Cluster"]]
plt.figure(figsize=(8, 6))
plt.scatter(df["Glucose"], df["BMI"], c=colors, alpha=0.5, edgecolors="k", linewidths=0.3)
plt.xlabel("Glucose")
plt.ylabel("BMI")
plt.title("K-Means Clusters (K=2): Glucose vs BMI")
plt.legend(handles=[
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="blue", markersize=10, label="Cluster 0"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="red",  markersize=10, label="Cluster 1")
])
plt.tight_layout()
plt.show()

In [ ]:
centers = scaler.inverse_transform(kmeans.cluster_centers_)
center_df = pd.DataFrame(centers, columns=feature_cols)
center_df.index.name = "Cluster"
print("Cluster Centers (original scale):")
print(center_df.round(2))